In [1]:
import pandas as pd, xarray as xr, numpy as np, matplotlib as mpl

from functions import areaweighted_mean, reorient_netcdf

In [2]:
n_members = 500

## File path shortcuts
basepath = '/home/msaenger/Perceptions_2025/'
cpath = basepath + 'CIDER_scenarios/'
vpath = '/home/msaenger/InVERT_output/'
newpath = basepath + f'cleaned_code/{n_members}_members/'

#### Load InVERT output (5 x 100-member ensembles)

In [4]:
invert1 = xr.open_dataset(vpath + '05.19.2026_SSP370_100member_ensemble_1/' + 'InVERT_100modes_1.nc')
invert2 = xr.open_dataset(vpath + '05.19.2026_SSP370_100member_ensemble_2/' + 'InVERT_100modes_2.nc')
invert3 = xr.open_dataset(vpath + '05.19.2026_SSP370_100member_ensemble_3/' + 'InVERT_100modes_3.nc')
invert4 = xr.open_dataset(vpath + '05.19.2026_SSP370_100member_ensemble_4/' + 'InVERT_100modes_4.nc')
invert5 = xr.open_dataset(vpath + '05.19.2026_SSP370_100member_ensemble_5/' + 'InVERT_100modes_5.nc')

In [5]:
invert_all = xr.concat([invert1, invert2, invert3, invert4, invert5], dim='ensemble')
invert_all

<xarray.Dataset> Size: 44GB
Dimensions:   (ensemble: 500, time: 600, lat: 96, lon: 192)
Coordinates:
    month     (time) int64 5kB 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12
  * lat       (lat) float64 768B -88.57 -86.72 -84.86 ... 84.86 86.72 88.57
  * lon       (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
  * time      (time) float64 5kB 0.0 1.0 2.0 3.0 4.0 ... 596.0 597.0 598.0 599.0
  * ensemble  (ensemble) int64 4kB 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
Data variables:
    T         (ensemble, time, lat, lon) float64 44GB 0.135 0.1382 ... -2.563
    gmean     (ensemble, time) float64 2MB 0.05775 0.06329 ... 0.051 -0.1232

#### Load SSP370 ensemble mean TREFHT

In [6]:
SSP = xr.open_mfdataset(basepath + 'SSP370_preprocessing_data/SSP370_full_ensemble_TREFHT_regridded.nc')
GHG_background = SSP.emean
GHG_background

<xarray.DataArray 'emean' (time: 1032, lat: 96, lon: 192)> Size: 76MB
dask.array<open_dataset-emean, shape=(1032, 96, 192), dtype=float32, chunksize=(1, 96, 192), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) object 8kB 2015-01-16 00:00:00 ... 2100-12-16 00:00:00
  * lon      (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
  * lat      (lat) float64 768B -88.57 -86.72 -84.86 -83.0 ... 84.86 86.72 88.57

#### Combine SSP370 forced response and InVERT to create an ensemble

In [7]:
## Prepare datasets to combine: make sure they are the same length
## InVERT ensemble used here has 50 years of variability data

## Select the first 50 years of SSP370 data
GHG_background_short = GHG_background.isel(time=slice(0, len(invert_all.time)))
GHG_background_short

<xarray.DataArray 'emean' (time: 600, lat: 96, lon: 192)> Size: 44MB
dask.array<getitem, shape=(600, 96, 192), dtype=float32, chunksize=(1, 96, 192), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) object 5kB 2015-01-16 00:00:00 ... 2064-12-16 00:00:00
  * lon      (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
  * lat      (lat) float64 768B -88.57 -86.72 -84.86 -83.0 ... 84.86 86.72 88.57

In [8]:
invert_all['time'] = GHG_background_short.time
invert_all

## Add variability output to GHG forced-response data
ssp_invert_ensemble = GHG_background_short + invert_all.['T']
ssp_invert_ensemble

<xarray.DataArray (time: 600, lat: 96, lon: 192, ensemble: 500)> Size: 44GB
dask.array<add, shape=(600, 96, 192, 500), dtype=float64, chunksize=(1, 96, 192, 500), chunktype=numpy.ndarray>
Coordinates:
  * time      (time) object 5kB 2015-01-16 00:00:00 ... 2064-12-16 00:00:00
  * lon       (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
  * lat       (lat) float64 768B -88.57 -86.72 -84.86 ... 84.86 86.72 88.57
    month     (time) int64 5kB 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12
  * ensemble  (ensemble) int64 4kB 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99

In [9]:
ssp_invert_ensemble = ssp_invert_ensemble.to_dataset(name='T')
ssp_invert_ensemble['gmean'] = areaweighted_mean(ssp_invert_ensemble.T)
ssp_invert_ensemble

<xarray.Dataset> Size: 44GB
Dimensions:   (time: 600, lon: 192, lat: 96, ensemble: 500)
Coordinates:
  * time      (time) object 5kB 2015-01-16 00:00:00 ... 2064-12-16 00:00:00
  * lon       (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
  * lat       (lat) float64 768B -88.57 -86.72 -84.86 ... 84.86 86.72 88.57
    month     (time) int64 5kB 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12
  * ensemble  (ensemble) int64 4kB 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
Data variables:
    T         (time, lat, lon, ensemble) float64 44GB dask.array<chunksize=(1, 96, 192, 500), meta=np.ndarray>
    gmean     (time, ensemble) float64 2MB dask.array<chunksize=(1, 500), meta=np.ndarray>

In [10]:
## Check months are aligned
print(GHG_background_short['time.month'].values == ssp_invert_ensemble['time.month'].values)
print(invert_all['time.month'].values == ssp_invert_ensemble['time.month'].values)

[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  T

In [11]:
## Update longitude coords from 0-360 to (-180 to 180) 
ssp_plus_invert = reorient_netcdf(ssp_invert_ensemble).rename({'T':'TREFHT'})
ssp_plus_invert

<xarray.Dataset> Size: 44GB
Dimensions:   (time: 600, lat: 96, ensemble: 500, lon: 192)
Coordinates:
  * time      (time) object 5kB 2015-01-16 00:00:00 ... 2064-12-16 00:00:00
  * lat       (lat) float64 768B -88.57 -86.72 -84.86 ... 84.86 86.72 88.57
    month     (time) int64 5kB 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12
  * ensemble  (ensemble) int64 4kB 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
  * lon       (lon) float64 2kB -178.1 -176.2 -174.4 ... 176.2 178.1 180.0
Data variables:
    TREFHT    (time, lat, lon, ensemble) float64 44GB dask.array<chunksize=(1, 96, 192, 500), meta=np.ndarray>
    gmean     (time, ensemble) float64 2MB dask.array<chunksize=(1, 500), meta=np.ndarray>

In [13]:
## Change ensemble values to go from 0 to len(ds.ensemble)

ssp_plus_invert['ensemble'] = np.arange(0, len(ssp_plus_invert.ensemble))
ssp_plus_invert

<xarray.Dataset> Size: 44GB
Dimensions:   (time: 600, lat: 96, ensemble: 500, lon: 192)
Coordinates:
  * time      (time) object 5kB 2015-01-16 00:00:00 ... 2064-12-16 00:00:00
  * lat       (lat) float64 768B -88.57 -86.72 -84.86 ... 84.86 86.72 88.57
    month     (time) int64 5kB 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12
  * ensemble  (ensemble) int64 4kB 0 1 2 3 4 5 6 ... 493 494 495 496 497 498 499
  * lon       (lon) float64 2kB -178.1 -176.2 -174.4 ... 176.2 178.1 180.0
Data variables:
    TREFHT    (time, lat, lon, ensemble) float64 44GB dask.array<chunksize=(1, 96, 192, 500), meta=np.ndarray>
    gmean     (time, ensemble) float64 2MB dask.array<chunksize=(1, 500), meta=np.ndarray>

In [14]:
ssp_plus_invert.to_netcdf(newpath + 'SSP370_plus_InVERT_500member_ensemble_TREFHT_2015_to_2065.nc')

### Next: calculate z-scores and then f at each grid cell, then aggregate f by country to find SAI start dates